In [21]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import mean_squared_error as mse, r2_score, mean_absolute_error as mae
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.preprocessing import MinMaxScaler
import torch.nn as nn
import random
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import torch.nn as nn

torch.set_default_dtype(torch.float64)
dtype = torch.get_default_dtype()
import warnings
warnings.filterwarnings('ignore')

In [22]:
df=pd.read_csv('Data.csv',header=0)

In [23]:
df=df.drop(columns=['id'])

In [24]:
state_encoder=LabelEncoder()
type_encoder=LabelEncoder()
df['state']=state_encoder.fit_transform(df['state'])
df['type']=type_encoder.fit_transform(df['type'])

In [25]:
label_mapping = dict(zip(state_encoder.classes_, state_encoder.transform(state_encoder.classes_)))
for label, encoded_value in label_mapping.items():
    print(f"'{label}' -> {encoded_value}")

'B' -> 0
'IR' -> 1
'N' -> 2
'OR' -> 3


In [26]:
label_mapping = dict(zip(type_encoder.classes_, type_encoder.transform(type_encoder.classes_)))
for label, encoded_value in label_mapping.items():
    print(f"'{label}' -> {encoded_value}")

'BA' -> 0
'DE' -> 1
'FE' -> 2


In [27]:
df

,state,type,rpm,load,t_mean,t_std,t_rms,t_skew,t_kurt,t_ptp,...,f_dominant_freq,f_5th_percentile,f_25th_percentile,f_75th_percentile,f_95th_percentile,f_wavelet_mean,f_wavelet_median,f_wavelet_rms,f_wavelet_std,f_wavelet_variance
0,0,0,1797,0,0.002853,0.017311,0.017545,0.014239,0.075489,0.134174,...,4139,0.006387,0.123569,0.738784,2.422547,0.734121,0.314228,1.921934,1.776203,3.154896
1,0,0,1797,0,0.002529,0.017200,0.017385,0.032497,0.058692,0.139369,...,4139,0.005994,0.119779,0.726986,2.482625,0.728453,0.305958,1.904439,1.759614,3.096243
2,0,0,1797,0,0.002602,0.017081,0.017278,0.061202,0.160682,0.151499,...,4139,0.008661,0.113296,0.699157,2.430051,0.712182,0.297548,1.892670,1.753567,3.074998
3,0,0,1797,0,0.002708,0.016975,0.017190,0.013459,0.195683,0.144623,...,4139,0.010605,0.120938,0.705810,2.373509,0.713211,0.309858,1.883057,1.742766,3.037233
4,0,0,1797,0,0.002684,0.016787,0.017000,0.033559,0.050973,0.133537,...,4139,0.020710,0.114973,0.687536,2.396269,0.707328,0.294288,1.862265,1.722706,2.967717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1175,2,2,1730,3,0.015006,0.036903,0.039838,0.009523,-0.035053,0.271790,...,0,0.032399,0.126923,0.907898,4.920922,1.221925,0.288156,4.363985,4.189422,17.551261
1176,2,2,1730,3,0.015710,0.036018,0.039295,-0.008692,0.096287,0.312720,...,0,0.014344,0.118182,0.907908,4.909120,1.202052,0.275093,4.304556,4.133313,17.084274
1177,2,2,1730,3,0.015591,0.035598,0.038862,0.034300,-0.015876,0.270019,...,0,0.013124,0.115520,0.897006,4.866387,1.187265,0.270609,4.257128,4.088220,16.713546
1178,2,2,1730,3,0.014865,0.035745,0.038713,-0.047950,-0.025512,0.265286,...,0,0.037472,0.118846,0.917848,4.721820,1.201819,0.277128,4.240771,4.066911,16.539767


In [32]:
from kan import KAN

In [33]:
model = KAN(width=[33,20,4], grid=5, k=6, seed=42)

checkpoint directory created: ./model
saving model version 0.0


In [35]:
x = df.drop(columns='state')
y = df['state']
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=26)

In [37]:
x, y = smote.fit_resample(x, y)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=26)

In [38]:
dataset = {}
train_input=x_train.to_numpy()
train_label=y_train.to_numpy()
test_input=x_test.to_numpy()
test_label=y_test.to_numpy()

In [39]:
scaler_input = MinMaxScaler()

train_input_scaled = scaler_input.fit_transform(train_input)
test_input_scaled = scaler_input.transform(test_input)

dataset['train_input'] = torch.tensor(train_input_scaled, dtype=dtype)  # Convert to the specified dtype
dataset['test_input'] = torch.tensor(test_input_scaled, dtype=dtype)    # Convert to the specified dtype

# Reshape train_label and test_label to match the expected shape (for a classification task)
dataset['train_label'] = torch.tensor(train_label[:, None], dtype=dtype)  # Ensure shape is (n_samples, 1)
dataset['test_label'] = torch.tensor(test_label[:, None], dtype=dtype)    # Ensure shape is (n_samples, 1)

In [40]:
model(dataset['train_input'])

tensor([[-0.0266,  0.0815, -0.0415, -0.0688],
        [-0.0170,  0.0348, -0.0015, -0.0441],
        [ 0.0012,  0.0614, -0.0334, -0.0549],
        ...,
        [ 0.0019,  0.0466, -0.0278, -0.0533],
        [-0.0274,  0.1027, -0.0382, -0.0901],
        [-0.0674,  0.1596, -0.0544, -0.1090]], grad_fn=<AddBackward0>)

In [ ]:
model.plot()